# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description
pp = pprint.PrettyPrinter()
md = dataset.metadata
print(f"{md.name}: {md.description}")
print(f"\nPublished: {md.date_published}")
print(f"License: {md.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

In [ ]:
# List all available record sets and their field structure.
print('Available record sets (by @id):')
record_set_ids = [rs['@id'] for rs in dataset.metadata._json_ld.get('recordSet', [])]
for rs in dataset.metadata._json_ld.get('recordSet', []):
    print(f"\nRecordSet: {rs['@id']}")
    if 'name' in rs:
        print(f"  Name: {rs['name']}")
    if 'field' in rs:
        print("  Fields and columns:")
        for f in rs['field']:
            if isinstance(f, dict):  # Inline field
                print(f"    Field @id: {f.get('@id', 'N/A')}, name: {f.get('name','N/A')}")
                if 'column' in f:
                    columns = f['column']
                    if isinstance(columns, list):
                        for c in columns:
                            if isinstance(c, dict):
                                print(f"      Column @id: {c.get('@id', 'N/A')}, name: {c.get('name','N/A')}")
                            else:
                                print(f"      Column @id: {c}")
                    else:
                        print(f"      Column @id: {columns}")
            else:
                print(f"    Field @id: {f}")

> The Croissant schema provides structured metadata. To enumerate fields and columns, we used the `@id` for linkage. 

### Example Record Preview
Let's examine the records of the first available record set by `@id`.

In [ ]:
# Pick the first record set for preview
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"\nPreviewing records for record set '@id': {first_rs}\n---")
    cnt = 0
    for rec in dataset.records(record_set=first_rs):
        print(rec)
        cnt += 1
        if cnt > 2:
            break
else:
    print("No record sets found in the dataset.")

## 3. Data Extraction
Load data from each record set into a DataFrame for further analysis.

We use record set and field `@id` values as references.

In [ ]:
# Extract data from each record set (@id)
dataframes = {}
if not record_set_ids:
    print("No record sets to extract.")
else:
    for rsid in record_set_ids:
        print(f"Loading record set: {rsid}")
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        print(f"  -> loaded {len(df)} records." if not df.empty else "  (empty)")
        dataframes[rsid] = df
        if not df.empty:
            print(f"  Columns: {df.columns.tolist()}")

# Example: Display first 5 rows for the first record set (if any records exist)
if record_set_ids:
    rs_view = record_set_ids[0]
    if rs_view in dataframes and not dataframes[rs_view].empty:
        display_cols = dataframes[rs_view].columns.tolist()
        print(f"Sample rows from record set {rs_view}:")
        display(dataframes[rs_view].head())
    else:
        print(f"No data found for record set {rs_view}")

## 4. Exploratory Data Analysis (EDA)
Apply filtering, normalization, and grouping to the DataFrame for a numeric field and a group attribute.

> **Reference fields by `@id`. Adjust the code to use actual field IDs as discovered in the previous steps.**

In [ ]:
####################
# Configuration: set field @id values here (USE THE @id values printed in data overview above):
# Example:
#   numeric_field = 'cr:Field_accession_coverage'   
#   group_field = 'cr:Field_accession_sample_id'

## Set your field @id from the record set
record_set_id = record_set_ids[0] if record_set_ids else None
if not record_set_id or dataframes[record_set_id].empty:
    print("No data for EDA!")
else:
    # Attempt to detect a numeric field for demonstration
    df = dataframes[record_set_id]
    possible_numeric = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not possible_numeric:
        print("No numeric field found for EDA.")
    else:
        numeric_field = possible_numeric[0]  # Use the first numeric column found
        print(f'Using numeric field (@id): {numeric_field}')
        threshold = df[numeric_field].quantile(0.75) if not df[numeric_field].isnull().all() else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        col_norm = f"{numeric_field}_normalized"
        filtered_df[col_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std(ddof=0)
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, col_norm]].head())

        # Try to group by the first non-numeric field
        non_numeric = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
        if non_numeric:
            group_field = non_numeric[0]
            print(f"Grouping by field (@id): {group_field}")
            grouped_df = (
                filtered_df.groupby(group_field)[numeric_field].mean().reset_index().rename({numeric_field: f"mean_{numeric_field}"}, axis=1)
            )
            print("Mean by group:")
            display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids:
    rsid = record_set_ids[0]
    df = dataframes[rsid]
    if not df.empty:
        num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        if num_cols:
            col = num_cols[0]
            plt.figure(figsize=(8,4))
            sns.histplot(df[col].dropna(), bins=30, kde=True)
            plt.title(f'Distribution of {col} (@id) in {rsid}')
            plt.xlabel(col)
            plt.show()

        # If at least two numeric fields, show a pairplot
        if len(num_cols) > 1:
            sns.pairplot(df[num_cols].dropna())
            plt.suptitle('Pairplot of numeric columns')
            plt.show()
        
        # Optionally: bar chart for non-numeric group field
        non_num = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
        if non_num:
            group_field = non_num[0]
            plt.figure(figsize=(8,4))
            sns.countplot(y=df[group_field])
            plt.title(f'Counts by {group_field} (@id)')
            plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we've demonstrated how to fetch and analyze the FAIR\^2 Croissant dataset using only official metadata relationships (`@id`).

* Fields and columns are always referenced by their unique Croissant `@id`.
* You can extend this notebook by selecting other record sets and fields for more specific analyses.
* Use the code blocks in sections 4 and 5 as templates for more advanced processing or visualization.

For full documentation on the Croissant metadata model and the mlcroissant library, visit: https://mlcommons.github.io/croissant/.